# HS4002 Week 11

## Panel Data Models

Panel data follows the **same units** (people, players, countries) across multiple time points, allowing us to control for unobserved unit-level heterogeneity.

In Python, the `linearmodels` library provides panel regression — equivalent to R's `plm` package.

We'll use NBA player data (2013–2017) to ask: **how does a player's age affect their scoring performance?**

In [1]:
# !pip install pandas numpy plotnine linearmodels stargazer

%pip install linearmodels stargazer plotnine
import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup: Load and Prepare the NBA Dataset

In [2]:
# Data source: https://www.kaggle.com/datasets/drgilermo/nba-players-stats
# NBA player statistics 1950-2017 from basketball-reference.com
raw_df = pd.read_csv("bball.csv").iloc[:, 1:]  # drop the unnamed index column

df = (
	raw_df[raw_df["Year"] >= 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"])
	.reset_index()
)
print(df.shape)
df.head()

(2403, 23)


,Player,Year,games_played,games_started,minutes_played,age,personal_fouls,points,true_shooting,assists,...,fg,threeP,threePA,twoP,twoPA,ft,fta,def_rb,off_rb,ppg
0,A.J. Hammons,2017.0,22.0,0.0,163.0,24.0,21.0,48.0,0.472000,4.0,...,17.0,5.0,10.0,12.0,32.0,9.0,20.0,28.0,8.0,2.181818
1,A.J. Price,2013.0,57.0,22.0,1278.0,26.0,73.0,441.0,0.501000,205.0,...,161.0,70.0,200.0,91.0,213.0,49.0,62.0,94.0,20.0,7.736842
2,A.J. Price,2014.0,28.0,0.0,99.0,27.0,5.0,44.0,0.469000,13.0,...,19.0,6.0,22.0,13.0,24.0,0.0,2.0,9.0,1.0,1.571429
3,A.J. Price,2015.0,52.0,0.0,648.0,28.0,30.0,266.0,0.376750,92.0,...,102.0,30.0,114.0,72.0,160.0,32.0,48.0,52.0,12.0,5.115385
4,Aaron Brooks,2013.0,106.0,40.0,1994.0,28.0,188.0,752.0,0.500667,228.0,...,286.0,100.0,268.0,186.0,364.0,80.0,104.0,134.0,26.0,7.094340


## Setting up the Panel Index

`linearmodels` requires a **MultiIndex** of (entity, time). Equivalent to the `index=c('Player','Year')` argument in R's `plm()`.

In [3]:
# Set the panel index — required by linearmodels
#need to tell it what is the entity and which is the time column 
df_panel = df.set_index(['Player', 'Year'])

# A Tutorial for Panel Models

## Pooled OLS

Equivalent to a regular `lm()` in R — ignores the panel structure.

In [4]:
#simple linear, OLS model 
pool_model = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=df_panel)
pool_result = pool_model.fit()
print(pool_result.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6757
Estimator:                  PooledOLS   R-squared (Between):              0.6103
No. Observations:                2403   R-squared (Within):               0.0318
Date:                Thu, Nov 05 2026   R-squared (Overall):              0.6757
Time:                        06:45:55   Log-likelihood                   -7587.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2501.2
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2501.2
                            

## Fixed Effects (Within) Model

Adds `EntityEffects` to control for time-invariant player characteristics. Equivalent to `plm(..., model='within')` in R.

In [5]:
fe_model = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=df_panel)
fe_result = fe_model.fit()
print(fe_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Thu, Nov 05 2026   R-squared (Overall):             -43.473
Time:                        06:45:55   Log-likelihood                   -5167.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             170.80
                            

Interpret the coefficients. How does the fixed-effects estimate differ from the pooled OLS estimate?

## Random Effects Model

Equivalent to `plm(..., model='random')` in R.

In [6]:
re_model = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=df_panel)
re_result = re_model.fit()
print(re_result.summary)


                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3804
Estimator:              RandomEffects   R-squared (Between):              0.6371
No. Observations:                2403   R-squared (Within):               0.0412
Date:                Thu, Nov 05 2026   R-squared (Overall):              0.6573
Time:                        06:45:55   Log-likelihood                   -5810.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      737.02
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             737.02
                            

## Model Diagnostics

### Lagrange Multiplier Test

Tests whether individual effects are present (i.e., whether FE/RE is needed over pooled OLS). Equivalent to R's `plmtest(fe_model, effect='individual')`.

In [7]:
# Breusch-Pagan LM test for random effects
# Under H0 (no individual effects), we prefer pooled OLS.
# Equivalent to R's plmtest(..., effect='individual', type='bp').
from scipy.stats import chi2

resid = pool_result.resids
e_i   = resid.groupby(level=0).sum()
ssr   = float((resid ** 2).sum())
A1    = float((e_i ** 2).sum()) / ssr - 1
T_i   = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects:')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

LM test for individual effects:
LM statistic = 2381.2182
df = 1
p-value = 0.0000


### Hausman Test

Tests whether RE estimates are consistent (i.e., whether FE or RE is preferred). Under H0, RE is consistent and more efficient. Equivalent to R's `phtest(fe_model, re_model)`.

In [8]:
# Hausman test: compare FE and RE estimates
# Under H0 (RE consistent), the difference in coefficients should be small
from scipy.stats import chi2

common_params = fe_result.params.index.intersection(re_result.params.index)

beta_diff = fe_result.params[common_params] - re_result.params[common_params]
cov_diff  = (
	fe_result.cov.loc[common_params, common_params] -
	re_result.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val  = chi2.sf(h_stat, df=len(common_params))
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

Hausman statistic = 285.0094
df = 2
p-value = 0.0000
Prefer FE over RE: True


## Comparing the Models

As in earlier weeks, `stargazer` collects the fitted models into one regression table. Reading pooled OLS, fixed effects, and random effects side by side is easier than scrolling through three summaries. Equivalent to R's `stargazer(lm1, fe_glm1, re_glm1)`.

In [9]:
from stargazer.stargazer import Stargazer

sg = Stargazer([pool_result, fe_result, re_result])
sg.title('Points per Game and Age')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Points per game')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg.covariate_order(['age', 'I(age ** 2)'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
})

# Ending the cell on the object itself renders the table in the notebook
sg

# Independent Analysis: Has Sports Science Extended Career Peaks?
Compare players from 2008–2012 with players from 2013–2017.
 Does age affect scoring differently across the two eras?

if you believe that sport sci have advanced such that player can perform even wben they are older (basic inuition of this^^)

In [17]:
def make_period_data(raw_df, start_year, end_year):
    return (
        raw_df[raw_df["Year"].between(start_year, end_year)]
        .groupby(["Player", "Year"])
        .agg(
            games_played=("G", "sum"),
            games_started=("GS", "sum"),
            minutes_played=("MP", "sum"),
            age=("Age", "mean"),
            personal_fouls=("PF", "sum"),
            points=("PTS", "sum"),
            true_shooting=("TS%", "mean"),
            assists=("AST", "sum"),
            steals=("STL", "sum"),
            blocks=("BLK", "sum"),
            fga=("FGA", "sum"),
            fg=("FG", "sum"),
            threeP=("3P", "sum"),
            threePA=("3PA", "sum"),
            twoP=("2P", "sum"),
            twoPA=("2PA", "sum"),
            ft=("FT", "sum"),
            fta=("FTA", "sum"),
            def_rb=("DRB", "sum"),
            off_rb=("ORB", "sum"),
        )
        .assign(ppg=lambda x: x["points"] / x["games_played"])
        .reset_index()
    )


df_2008_2012 = make_period_data(raw_df, 2008, 2012)
df_2013_2017 = make_period_data(raw_df, 2013, 2017)

print("2008-2012:", df_2008_2012.shape)
print("2013-2017:", df_2013_2017.shape)


2008-2012: (2266, 23)
2013-2017: (2403, 23)


In [18]:
# (2) fit models for each of the two data sets
panel_2008_2012 = df_2008_2012.set_index(["Player", "Year"])
panel_2013_2017 = df_2013_2017.set_index(["Player", "Year"])




# For 2008-2012 analysis

In [24]:
# (3) run diagnostics, compare, and interpret for 2008-2012.

# Pooled OLS
pool_model_early = PooledOLS.from_formula("ppg ~ 1 + age + I(age**2)", data=panel_2008_2012)
pool_result_early = pool_model_early.fit(cov_type="clustered", cluster_entity=True)
print(pool_result_early.summary)



                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.0156
Estimator:                  PooledOLS   R-squared (Between):             -0.0760
No. Observations:                2266   R-squared (Within):               0.0941
Date:                Thu, Nov 05 2026   R-squared (Overall):              0.0156
Time:                        07:24:39   Log-likelihood                   -7213.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      17.934
Entities:                         743   P-value                           0.0000
Avg Obs:                       3.0498   Distribution:                  F(2,2263)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             9.0483
                            

In [25]:
# Fixed effects
fe_model_early = PanelOLS.from_formula("ppg ~ age + I(age**2) + EntityEffects", data=panel_2008_2012)
fe_result_early = fe_model_early.fit(cov_type="clustered", cluster_entity=True)
print(fe_result_early.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.2139
Estimator:                   PanelOLS   R-squared (Between):             -39.949
No. Observations:                2266   R-squared (Within):               0.2139
Date:                Thu, Nov 05 2026   R-squared (Overall):             -26.718
Time:                        07:24:40   Log-likelihood                   -4814.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      206.98
Entities:                         743   P-value                           0.0000
Avg Obs:                       3.0498   Distribution:                  F(2,1521)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             108.97
                            

In [26]:
# Random effects
re_model_early = RandomEffects.from_formula("ppg ~ 1 + age + I(age**2)", data=panel_2008_2012)
re_result_early = re_model_early.fit(cov_type="clustered", cluster_entity=True)
print(re_result_early.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.0409
Estimator:              RandomEffects   R-squared (Between):             -0.1060
No. Observations:                2266   R-squared (Within):               0.1837
Date:                Thu, Nov 05 2026   R-squared (Overall):             -0.0665
Time:                        07:24:41   Log-likelihood                   -5338.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      131.33
Entities:                         743   P-value                           0.0000
Avg Obs:                       3.0498   Distribution:                  F(2,2263)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             77.217
                            

# For 2013-2017 analysis

In [27]:
# (4) run diagnostics, compare, and interpret for 2013-2017.

# Pooled OLS
pool_model_late = PooledOLS.from_formula("ppg ~ 1 + age + I(age**2)", data=panel_2013_2017)
pool_result_late = pool_model_late.fit(cov_type="clustered", cluster_entity=True)
print(pool_result_late.summary)


                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.0169
Estimator:                  PooledOLS   R-squared (Between):             -0.0957
No. Observations:                2403   R-squared (Within):               0.0852
Date:                Thu, Nov 05 2026   R-squared (Overall):              0.0169
Time:                        07:25:01   Log-likelihood                   -7578.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      20.673
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2400)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             9.8063
                            

In [28]:
# Fixed effects
fe_model_late = PanelOLS.from_formula("ppg ~ age + I(age**2) + EntityEffects", data=panel_2013_2017)
fe_result_late = fe_model_late.fit(cov_type="clustered", cluster_entity=True)
print(fe_result_late.summary)



                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Thu, Nov 05 2026   R-squared (Overall):             -43.473
Time:                        07:25:02   Log-likelihood                   -5167.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             96.728
                            

In [29]:
# Random effects
re_model_late = RandomEffects.from_formula("ppg ~ 1 + age + I(age**2)", data=panel_2013_2017)
re_result_late = re_model_late.fit(cov_type="clustered", cluster_entity=True)
print(re_result_late.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                       -0.0029
Estimator:              RandomEffects   R-squared (Between):             -0.0837
No. Observations:                2403   R-squared (Within):               0.1505
Date:                Thu, Nov 05 2026   R-squared (Overall):             -0.0531
Time:                        07:25:02   Log-likelihood                   -5736.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      107.01
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2400)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             78.541
                            

# Hausman test 

In [ ]:
def hausman_test(panel):
    """Hausman test: random effects vs fixed effects."""
    fe = PanelOLS.from_formula("ppg ~ age + I(age**2) + EntityEffects", data=panel).fit()
    re = RandomEffects.from_formula("ppg ~ 1 + age + I(age**2)", data=panel).fit()

    common = fe.params.index.intersection(re.params.index)
    b_diff = (fe.params[common] - re.params[common]).to_numpy()
    V_diff = (fe.cov.loc[common, common] - re.cov.loc[common, common]).to_numpy()

    if (np.linalg.eigvalsh(V_diff) <= 0).any():
        print("Warning: V_diff is not positive definite, so this result is unreliable.")

    H = b_diff @ np.linalg.inv(V_diff) @ b_diff
    return H, chi2.sf(H, df=len(common))
    


In [36]:
h_early, h_p_early = hausman_test(panel_2008_2012)
print(f"Hausman test (2008-2012): {h_early:.2f}, p = {h_p_early:.4f}")

Hausman test (2008-2012): 151.69, p = 0.0000


In [37]:
h_late, h_p_late = hausman_test(panel_2013_2017)
print(f"Hausman test (2013-2017): {h_late:.2f}, p = {h_p_late:.4f}")

Hausman test (2013-2017): 131.39, p = 0.0000


# Lagrange test 

In [ ]:
def bp_lm_test(panel):
    """Breusch-Pagan LM test: pooled OLS vs random effects."""
    pooled = PooledOLS.from_formula("ppg ~ 1 + age + I(age**2)", data=panel).fit()

    e = pooled.resids
    sum_e = e.groupby(level=0).sum()   # total residual per player
    T = e.groupby(level=0).size()      # observations per player
    n = T.sum()

    A = (sum_e**2).sum() / (e**2).sum() - 1
    lm = n**2 / (2 * ((T**2).sum() - n)) * A**2
    return lm, chi2.sf(lm, df=1)


In [34]:
lm_early, lm_p_early = bp_lm_test(panel_2008_2012)
print(f"LM test (2008-2012): {lm_early:.2f}, p = {lm_p_early:.4f}")

LM test (2008-2012): 2494.25, p = 0.0000


In [35]:
lm_late, lm_p_late = bp_lm_test(panel_2013_2017)
print(f"LM test (2013-2017): {lm_late:.2f}, p = {lm_p_late:.4f}")


LM test (2013-2017): 2429.25, p = 0.0000


# Intepretation of results

In [14]:
# (5) what differences do you find in the age-scoring relationship across the two eras?

Based on the Hausman test and lagrange test results, the fixed model should be used for comparision. 

*_Peak age calulation_*
Younger age group (2013-2017) = 5.7447/(0.1078*2)= 26.6
Older age group (2008-2012) = 5.0941/(0.1021*2)=24.9 

Since the peak age of those in the later age group is higher, suggesting that players can not only perform but peak at older ages.
